In [ ]:
import saann.backend as BE
from saann.transformer.transformer_model import TransformerModel
from saann.generation import generate_top_p
from saann.training import train_transformer, load_model, create_optimizer
from saann.tokenizer import CharTokenizer, ByteTokenizer

In [ ]:
# Gather text to train on
text = "This is a dummy example. The model will not learn anything useful. This is just to show the API of SaANN."

In [3]:
# Create the Tokenizer (either Byte or Char)
tokenizer = ByteTokenizer() #OR tokenizer = CharTokenizer(text)

In [ ]:
# Create model
vocab_size = tokenizer.vocab_size
embed_dim = 512
num_heads = 8
ff_hidden_dim = 2048
num_layers = 8
max_seq_len = 512

model = TransformerModel(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_hidden_dim=ff_hidden_dim,
    num_layers=num_layers,
    max_seq_len=max_seq_len,
    learned_positional=True
)

In [5]:
# Create optimizer
optimizer = create_optimizer(model)

In [6]:
# Process data
encoded_text = tokenizer.encode(text)
data = BE.xp.array(encoded_text, dtype=BE.xp.int32)
split = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]

len_input = len(train_data)

In [7]:
# Train model
train_transformer(model=model,
                  optimizer=optimizer,
                  data=train_data,
                  batch_size=32,
                  seq_len=64,
                  epochs=30,
                  checkpoint_every = 10,
                  checkpoint_dir="checkpoints",
                  tokenizer=tokenizer)

Time for first epoch: 1.339837 seconds. Estimated time: 0 minutes.
Initial loss: 5.5439
10% - Loss: 5.2564 (LR=0.000098)
20% - Loss: 5.0671 (LR=0.000091)
30% - Loss: 4.8426 (LR=0.000080)
40% - Loss: 4.4907 (LR=0.000067)
50% - Loss: 4.0616 (LR=0.000053)
60% - Loss: 3.6811 (LR=0.000038)
70% - Loss: 3.4242 (LR=0.000026)
80% - Loss: 3.2801 (LR=0.000016)
90% - Loss: 3.2032 (LR=0.000011)
Final loss: 3.1695


In [ ]:
# Load tranined model (OPTIONAL)
model, optimizer, tokenizer, scheduler = load_model("checkpoints/checkpoint_final.npz") # load_model(path)

Loading model of version: 0.3.0


In [15]:
# Validate model
start = BE.xp.array([val_data], dtype=BE.xp.int32)
generated = generate_top_p(model=model, start_tokens=start, max_new_tokens=30, p=0.9, temperature=0.5, rep_penalty=1.2)
decoded_output = tokenizer.decode(generated[0].tolist())
print(f"Decoded output: {decoded_output}")

Decoded output: I of SaANN. nmunmlinudelri helnm aesreilt
